In [2]:
# !pip install --upgrade pip
!pip install qdrant-client transformers torch torchaudio librosa numpy musicbrainzngs
pip install syncedlyrics "qdrant-client[fastembed]>=1.14.2" mutagen

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 9.7 MB/s eta 0:00:00


In [4]:
from init_utils import FileProcessor, LyricsDB
from qdrant_client import QdrantClient, models

In [2]:
!docker run -d -p 6333:6333 -p 6334:6334 \
   -v "./qdrant_storage:/qdrant/storage:z" \
   qdrant/qdrant

60256759b3b9de32b89cdd666f7f626f19c3f659397cb482365651cb324dab26


docker: Error response from daemon: failed to set up container networking: driver failed programming external connectivity on endpoint dreamy_napier (8ceeb7c21ea7d5794936029c13cd7f7bace73b507834cc45487d8eb3af913695): Bind for 0.0.0.0:6333 failed: port is already allocated

Run 'docker run --help' for more information


In [5]:
qd_client = QdrantClient("http://localhost:6333")
fp = FileProcessor()

In [23]:
qd_client.delete_collection("test")

True

In [10]:
data = fp.process_folder("S:\Music\Jay-Z - Коллекция (20 релизов) (FLAC)")

Найдено файлов: 260, воркеров: 8


  0%|          | 0/260 [00:00<?, ?it/s]

An error occurred while searching for an LRC on Megalobiz
('Connection aborted.', ConnectionResetError(10054, 'Удаленный хост принудительно разорвал существующее подключение', None, 10054, None))


✗ Jay-Z — Anything / Jigga My Nigga / Girl’s Best Friend (Feat. Mashonda)


An error occurred while searching for an LRC on Megalobiz
('Connection aborted.', ConnectionResetError(10054, 'Удаленный хост принудительно разорвал существующее подключение', None, 10054, None))


✗ Jay-Z — 44 Fours (live)


An error occurred while searching for an LRC on Megalobiz
('Connection aborted.', ConnectionResetError(10054, 'Удаленный хост принудительно разорвал существующее подключение', None, 10054, None))


✗ Jay-Z — Fallin' (Feat. Bilal)

Готово: 248/248 текстов найдено → lyrics.json


In [22]:
db = LyricsDB(data, qd_client, "test")
db.fit()

Collection test has been successfully created


RuntimeException: [ONNXRuntimeError] : 6 : RUNTIME_EXCEPTION : Non-zero status code returned while running Add node. Name:'/encoder/layer.0/attention/self/Add' Status Message: E:\_work\1\s\onnxruntime\core\framework\bfc_arena.cc:358 onnxruntime::BFCArena::AllocateRawInternal Failed to allocate memory for requested buffer of size 4680749056


In [111]:
db.search("home")

[ScoredPoint(id='4ed26b23-b39d-40cf-87f4-034e4ce72e36', version=1, score=1.7742814, payload={'lyrics': "9:09\nYou gonna call it or am I?\nOne more time\nThis puppy love is out of line\nOne more slide\nAnd then we're back to real life\nOoh and I'm falling now but it's so wrong\nYou talk like a man and taste like the sun\nOoh and you lift your eyes up from the dust\nI knew just then I knew it was done\nI guess I want you more than I thought I did\nNow that I know that part of you's at home with him\nI guess I want you more than I thought I did\nNow that I know that part of you's not part of this\nSoft blue sky\nHelium balloons float up away\nBroad daylight\nBut we're sunflowers in the rain\nMy momma said their used to be white pyramids\nThey float above the sand they're slowly sinking in\nAre our foundations destined to keep crumbling\nJust 'cause we started this with zero innocence?\nI just can't build on something that begins like this\nIt's a blood diamond, flawless but for that one t

## Classic embeddings

In [16]:
# Create the collection with specified sparse vector parameters
client.create_collection(
    collection_name="lyrics-sparse",
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(
            modifier=models.Modifier.IDF,
        )
    }
)

True

In [17]:
# Send the points to the collection
client.upsert(
    collection_name="lyrics-sparse",
    points=[
        models.PointStruct(
            id=uuid.uuid4().hex,
            vector={
                "bm25": models.Document(
                    text=song["lyrics"], 
                    model="Qdrant/bm25",
                ),
            },
            payload={
                "lyrics": song["lyrics"],
                "title": song["title"],
                "artist": song["artist"],
                "album": song["album"],
            }
        )
        for song in song_text
    ]
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [18]:
def search(query: str, limit: int = 1) -> list[models.ScoredPoint]:
    results = client.query_points(
        collection_name="lyrics-sparse",
        query=models.Document(
            text=query,
            model="Qdrant/bm25",
        ),
        using="bm25",
        limit=limit,
        with_payload=True,
    )

    return results.points

In [21]:
results = search("Ferrari", limit = 5)
print(f"Найдено {len(results)} результатов")
if results:
    print(f"Самый релевантный результат: {results[0].payload['artist']} - {results[0].payload['title']}")

Найдено 5 результатов
Самый релевантный результат: Jay-Z - Money Ain’t A Thang (Feat. Jermaine Dupri)


Теперь добавляем dense вектора

In [14]:
client.delete_collection("lyrics-sparse-and-dense")

True

In [19]:
client.create_collection(
    collection_name="lyrics-sparse-and-dense",
    vectors_config={
        # Named dense vector for jinaai/jina-embeddings-v2-small-en
        "jina-small": models.VectorParams(
            size=512,
            distance=models.Distance.COSINE,
        ),
    },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(
            modifier=models.Modifier.IDF,
        )
    }
)

True

In [20]:
def upsert_in_batches(client, collection_name, points, batch_size=100):
    for i in range(0, len(points), batch_size):
        batch = points[i:i + batch_size]
        client.upsert(
            collection_name=collection_name,
            points=batch
        )
        print(f"Upserted {min(i + batch_size, len(points))}/{len(points)}")

In [22]:
upsert_in_batches(
    client, 
    collection_name="lyrics-sparse-and-dense",
    points=[
        models.PointStruct(
            id=uuid.uuid4().hex,
            vector={
                "jina-small": models.Document(
                    text=song["lyrics"],
                    model="jinaai/jina-embeddings-v2-small-en",
                ),
                "bm25": models.Document(
                    text=song["lyrics"], 
                    model="Qdrant/bm25",
                ),
            },
            payload={
                "lyrics": song["lyrics"],
                "title": song["title"],
                "artist": song["artist"],
                "album": song["album"],
            }
        )
        for song in song_text
    ]
    )

Upserted 100/5441
Upserted 200/5441
Upserted 300/5441
Upserted 400/5441
Upserted 500/5441
Upserted 600/5441
Upserted 700/5441
Upserted 800/5441
Upserted 900/5441
Upserted 1000/5441
Upserted 1100/5441
Upserted 1200/5441
Upserted 1300/5441
Upserted 1400/5441
Upserted 1500/5441
Upserted 1600/5441
Upserted 1700/5441
Upserted 1800/5441
Upserted 1900/5441
Upserted 2000/5441
Upserted 2100/5441
Upserted 2200/5441
Upserted 2300/5441
Upserted 2400/5441
Upserted 2500/5441
Upserted 2600/5441
Upserted 2700/5441
Upserted 2800/5441
Upserted 2900/5441
Upserted 3000/5441
Upserted 3100/5441
Upserted 3200/5441
Upserted 3300/5441
Upserted 3400/5441
Upserted 3500/5441
Upserted 3600/5441
Upserted 3700/5441
Upserted 3800/5441
Upserted 3900/5441
Upserted 4000/5441
Upserted 4100/5441
Upserted 4200/5441
Upserted 4300/5441
Upserted 4400/5441
Upserted 4500/5441
Upserted 4600/5441
Upserted 4700/5441
Upserted 4800/5441
Upserted 4900/5441
Upserted 5000/5441
Upserted 5100/5441
Upserted 5200/5441
Upserted 5300/5441
Up

In [23]:
def multi_stage_search(query: str, limit: int = 1) -> list[models.ScoredPoint]:
    results = client.query_points(
        collection_name="lyrics-sparse-and-dense",
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="jinaai/jina-embeddings-v2-small-en",
                ),
                using="jina-small",
                # Prefetch ten times more results, then
                # expected to return, so we can really rerank
                limit=(10 * limit),
            ),
        ],
        query=models.Document(
            text=query,
            model="Qdrant/bm25", 
        ),
        using="bm25",
        limit=limit,
        with_payload=True,
    )

    return results.points

In [77]:
results = multi_stage_search("ferrari", limit = 5)
print(f"Найдено {len(results)} результатов")
if results:
    print(f"Самый релевантный результат: {results[0].payload['artist']} - {results[0].payload['title']}")

Найдено 4 результатов
Самый релевантный результат: Roddy Ricch - Underdog


In [83]:
results[0].payload

{'lyrics': 'Slide out the garage (Roof)\n I got a hundred cars\n This for the underdogs\n The teller just asked me how I want the cash in\n I told him, "A hundred, large"\n I want them gold wraps\n Printed up, double R\n Big dog, big dog, big dog, roof\n Poppin\' out, sayin\' I ain\'t great, who?\n Biscayne drive, I cut off my roof\n You know how I\'m comin\', it\'s take two\n Tell a-, "Get out the way"\n Four by four, big Phantom (Shoo)\n They thought I was cancelled, I piped up and flipped the channel\n LST on the brim, they ain\'t think we\'d win, I just talked to my brother (Woo)\n Ain\'t safe when you- with the underdog, I might bite, this get rough (Roof)\n I\'m tryna have paper, Robert Craft and put a Tom Brady\n On a stab, pull up on Dre, talking-, he- with the wave\n Straight out of Compton\n Slide out the garage (Roof)\n I got a hundred cars\n This for the underdogs\n The teller just asked me how I want the cash in\n I told him, "A hundred, large"\n I want them gold wraps\n P

In [17]:
results[0].payload['title']

'Breaking News (Skit)'

In [25]:
def rrf_search(query: str, limit: int = 1) -> list[models.ScoredPoint]:
    results = client.query_points(
        collection_name="lyrics-sparse-and-dense",
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="jinaai/jina-embeddings-v2-small-en",
                ),
                using="jina-small",
                limit=(5 * limit),
            ),
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="Qdrant/bm25",
                ),
                using="bm25",
                limit=(5 * limit),
            ),
        ],
        # Fusion query enables fusion on the prefetched results
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        with_payload=True,
    )

    return results.points

In [84]:
result = rrf_search("ferrari")

In [90]:
result[5].payload

{'lyrics': "Yeah\n Getting it in from here to china\n Yo\n Alright let me just cool this right now quickly\n Super trippy riding through the gritty inner city\n Roll with the committee\n Handle your business or handle the pity\n All I see is lots of titties\n I know bunch of hippie chicks\n That's ready to show me tricks\n They doing the splits\n I'm all up in the mix\n A choir out the mist\n I'm taking trips\n I'm in the Ferrari looking sick\n I'm in the Ferrari looking slick\n Letting the engine rip\n And getting a tire\n A' grippin' ah slippin' and slidin'\n Turning the music up\n I'm vibing now I'm flying\n Lord strap me if I'm lying\n I ain't perfect but I'm trying\n Going super sire and buying\n Anything that catches my eye\n Cause I'm a provider getting it in from here to china\n It's so minor I'm a survivor, never retire\n I'm a black tiger ready to blaze to the fire, live wire\n Now I'm rolling through the shires\n Blazing the green to get me higher\n Now I'm inspired\n Puttin

## Custom embeddings

In [2]:
import torch
import librosa
import numpy as np
from pathlib import Path
from transformers import ClapModel, ClapProcessor
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ClapModel.from_pretrained("laion/larger_clap_music").to(device)
processor = ClapProcessor.from_pretrained("laion/larger_clap_music")
model.eval()

Loading weights:   0%|          | 0/555 [00:00<?, ?it/s]

ClapModel(
  (text_model): ClapTextModel(
    (embeddings): ClapTextEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): ClapTextEncoder(
      (layer): ModuleList(
        (0-11): 12 x ClapTextLayer(
          (attention): ClapTextAttention(
            (self): ClapTextSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ClapTextSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm):

In [147]:
client.delete_collection(collection_name='lyrics-clap-embed')

True

In [124]:
client.create_collection(collection_name='lyrics-clap-embed', vectors_config=VectorParams(size=512, distance=Distance.COSINE))

True

In [140]:
def get_clap_embedding(audio_path: str) -> np.ndarray:
    audio, sr = librosa.load(audio_path, sr=48000, mono=True)
    inputs = processor(audio=audio, sampling_rate=48000, return_tensors="pt").to(device)  # audios= !
    with torch.no_grad():
        outputs = model.audio_model(**inputs["input_features"].unsqueeze(0) 
                                    if False else 
                                    {"input_features": inputs["input_features"]})
        # Используем тот же projection что и get_audio_features
        embedding = model.audio_projection(outputs.pooler_output)
    return embedding.squeeze().cpu().numpy()

audio_files = list(Path(folder).rglob("*.mp3")) + list(Path(folder).rglob("*.flac")) + list(Path(folder).rglob("*.m4a"))

points = []
for idx, filepath in enumerate(tqdm(audio_files[:100])):
    try:
        embedding = get_clap_embedding(str(filepath))
        points.append(PointStruct(
            id=idx,
            vector=embedding.tolist(),
            payload={"filename": filepath.name, "path": str(filepath)},
        ))
        print(f"[{idx+1}/{len(audio_files)}] {filepath.name}")
    except Exception as e:
        print(f"Ошибка {filepath.name}: {e}")

  0%|          | 0/100 [00:00<?, ?it/s]

[1/5770] Curtis Mayfield - Move on Up (Single Edit).mp3
[2/5770] Estelle - American Boy (feat. Kanye West).mp3
[3/5770] Howlin' Wolf - Howlin' for My Darling.mp3
[4/5770] The_Chainsmokers_-_Dont_Let_Me_Down_47963194.mp3
[5/5770] 01 - Two Divided By Zero.mp3
[6/5770] 02 - West End Girls.mp3
[7/5770] 03 - Opportunities (Let's Make Lots Of Money).mp3
[8/5770] 04 - Love Comes Quickly.mp3
[9/5770] 05 - Suburbia.mp3
[10/5770] 06 - Opportunities (Let's Make Lots Of Money) [Reprise].mp3
[11/5770] 07 - Tonight Is Forever.mp3
[12/5770] 08 - Violence.mp3
[13/5770] 09 - I Want A Lover.mp3
[14/5770] 10 - Later Tonight.mp3
[15/5770] 11 - Why Don't We Live Together.mp3
[16/5770] 01 - A Man Could Get Arrested (12'' B-Side).mp3
[17/5770] 02 - Opportunities (Let's Make Lots Of Money) (Full Length 7'' Mix).mp3
[18/5770] 03 - In The Night.mp3
[19/5770] 04 - Opportunities (Let's Make Lots Of Money) (12'' Mix).mp3
[20/5770] 05 - Why Don't We Live Together (Original New York Mix).mp3
[21/5770] 06 - West End 

In [141]:
for i in range(0, len(points), 64):
    batch = points[i:i+64]
    try:
        client.upsert(collection_name="lyrics-clap-embed", points=batch)
        print(f"✅ Загружен batch {i//64 + 1} ({len(batch)} точек)")
    except Exception as e:
        print(f"❌ Ошибка upsert batch {i//64 + 1}: {e}")

✅ Загружен batch 1 (64 точек)
✅ Загружен batch 2 (36 точек)


In [142]:
def search_by_clap(query: str, limit: int = 3):
    inputs = processor(text=query, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.text_model(**inputs)
        embedding = model.text_projection(outputs.pooler_output)
    vec = embedding.squeeze().cpu().numpy().tolist()
    results = client.query_points(
        collection_name='lyrics-clap-embed',
        query=vec,
        limit=limit
    )
    return results

In [118]:
with torch.no_grad():
    v1 = model.get_text_features(**inputs1).pooler_output
    v2 = model.get_text_features(**inputs2).pooler_output
print(torch.allclose(v1, v2))  # False
print(v1.shape)  # размерность

False
torch.Size([1, 512])


In [146]:
search_by_clap("Death")

QueryResponse(points=[ScoredPoint(id=22, version=3, score=0.012664463, payload={'filename': '08 - Love Comes Quickly (Dance Mix).mp3', 'path': 'S:\\Music\\1986 - Please - Further Listening 1984-1986 (2018 Remastered Version)\\CD2\\08 - Love Comes Quickly (Dance Mix).mp3'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=64, version=4, score=0.0123875905, payload={'filename': '15 - The Battle (feat. Celph Titled).mp3', 'path': 'S:\\Music\\Fort Minor - 2023 - The Rising Tied (Deluxe Edition)\\15 - The Battle (feat. Celph Titled).mp3'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=35, version=3, score=0.011692425, payload={'filename': '08 - The Boy Is Mine.mp3', 'path': 'S:\\Music\\Ariana Grande - eternal sunshine (slightly deluxe) - 2024\\08 - The Boy Is Mine.mp3'}, vector=None, shard_key=None, order_value=None)])

In [115]:
search_by_clap("Azaz").points[0].payload['path']

AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'squeeze'

In [100]:
inputs = processor(text="cars", return_tensors="pt").to(device)
with torch.no_grad():
    text_embedding = model.get_text_features(**inputs)
vec = text_embedding.pooler_output.squeeze().cpu().numpy()

In [101]:
vec

array([-2.29267869e-02, -7.22438544e-02,  2.05817670e-02,  3.75333242e-02,
        5.14467247e-02, -2.66306233e-02, -5.10869958e-02,  3.76885496e-02,
        8.34651664e-02,  4.81622033e-02,  4.64378810e-03,  2.26792414e-02,
       -2.67129578e-03, -7.44642466e-02,  1.85070410e-02, -1.18518025e-01,
       -6.44272044e-02,  2.89134420e-02,  3.40240188e-02,  1.40287638e-01,
       -1.49613693e-02, -2.28504073e-02,  2.07907874e-02,  8.25718278e-04,
       -1.24954162e-02, -1.67133752e-02,  5.78570701e-02, -1.74261276e-02,
        4.77154888e-02,  7.92361796e-02, -1.14123188e-01, -4.98207696e-02,
        9.48578864e-03,  5.31476662e-02,  1.74292587e-02,  1.48075707e-02,
       -1.96138620e-02, -1.70176430e-03,  7.54122660e-02, -1.44941891e-02,
        1.72703341e-02, -1.66840963e-02, -2.37722639e-02, -7.61616975e-02,
       -2.74774786e-02, -5.98667050e-03, -6.11289230e-04, -3.10427323e-02,
       -8.17077979e-02,  3.94550152e-02, -2.47500371e-02, -4.88764495e-02,
       -5.92850596e-02,  

In [111]:
results = client.query_points(
    collection_name='lyrics-clap-embed',
    query=vec,
    limit=10
)

In [113]:
results.points[0]

ScoredPoint(id=4448, version=70, score=0.02625664, payload={'filename': '02 - Hush Hush.flac', 'path': 'S:\\Music\\The Pussycat Dolls - Hush Hush [Single]\\02 - Hush Hush.flac'}, vector=None, shard_key=None, order_value=None)

## RAG

In [ ]:
pip install pydantic-ai

In [ ]:
from pydantic_ai import Agent, RunContext

In [ ]:
def search(query: str, limit: int = 1) -> list[models.ScoredPoint]:
    results = client.query_points(
        collection_name="lyrics-sparse-and-dense",
        prefetch=[
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="jinaai/jina-embeddings-v2-small-en",
                ),
                using="jina-small",
                limit=(5 * limit),
            ),
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="Qdrant/bm25",
                ),
                using="bm25",
                limit=(5 * limit),
            ),
        ],
        # Fusion query enables fusion on the prefetched results
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        with_payload=True,
    )

    return results.points

In [ ]:
developer_prompt="""



""".strip()

In [ ]:
model = ''
chat_agent = Agent(  
    model,
    system_prompt=developer_prompt
)

In [ ]:
from typing import Dict


@chat_agent.tool
def search_tool(ctx: RunContext, query: str) -> Dict[str, str]:
    """
    Search the music lyrics database for relevant entries matching the query.

    Parameters
    ----------
    query : str
        The search query string provided by the user.

    Returns
    -------
    list
        A list of search results (up to 5), each containing relevance information 
        and associated output IDs.
    """
    print(f"search('{query}')")
    return search(query)

In [ ]:
user_question = "I just discovered the course. Can I join now?"
agent_run = await chat_agent.run(user_question)
print(agent_run.output)